In [80]:
import os
import re
import math
import json
import logging
import random
import numpy as np
import pandas as pd
import scipy.linalg # 用於 FID
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torchvision.models import inception_v3, Inception_V3_Weights
from scipy.optimize import linear_sum_assignment # 用於匈牙利演算法
from typing import Optional, Tuple, List, Dict, Any
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm

In [81]:
# ==============================================================================
# 組態設定
# ==============================================================================
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

CONFIG = {
    # --- 資料參數 ---
    "data_path": r"C:\thesis\code\Taipei_CF\all_merged.csv", # 資料路徑
    "H": 20, # 網格高度
    "W": 20, # 網格寬度
    "D": 1,  # 網格深度 (流量圖為1)

    # --- 模型參數 ---
    "image_channels": 1,      # 主要資料(流量圖)的通道數
    "condition_input_channels": 2, # 條件處理器接收的原始條件通道數 (小時網格 + 星期網格)
    "condition_encode_dim": 16, # 條件處理器輸出的特徵維度 (可調)
    "base_channels_unet": 64,   # UNet3D 的基礎通道數
    "unet_dropout_rate": 0.1,
    "time_emb_dim": 256,        # 時間嵌入維度
    "val_calculation_freq": 4, #驗證損失計算的頻率

    # --- DDPM 參數 ---
    "timesteps": 1000,          # 擴散時間步長
    "beta_start": 1e-4,
    "beta_end": 0.02,

    # --- 訓練參數 ---
    "epochs": 128, # 可調整
    "batch_size": 128, # 依 GPU 記憶體調整
    "lr": 1e-3, # 學習率
    "num_workers": 0, # DataLoader 工作執行緒 (Windows 建議 0, Linux 可 >0)
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42, # 隨機種子
    "weight_decay": 1e-5, # 優化器的權重衰減 
    "lr_scheduler_factor": 0.5, # ReduceLROnPlateau: 學習率降低因子
    "lr_scheduler_patience": 4,   # ReduceLROnPlateau: 多少個 epoch 驗證損失未改善則降低學習率
    "lr_scheduler_min_lr": 1e-6,  # ReduceLROnPlateau: 學習率下限
    "early_stopping_patience": 8, # 早停: 多少個 epoch 驗證損失未改善則停止訓練 
    "lr_reductions_before_val_decision": 3, # 學習率調降幾次後，才開始用驗證損失做早停和模型選擇的決策
    "resume_from_checkpoint": True,  # 是否嘗試從檢查點恢復訓練
    "checkpoint_path": "best_ddpm_model_during_training.pth", #預設使用的檢查點檔案名

    # --- 評估參數 ---
    "eval_batch_size": 32,
    "fid_batch_size": 64,
    "fid_num_samples": 128, # FID 計算樣本數

    # --- 路徑與儲存 ---
    "save_dir": "results_ddpm_conditioned_flow_taipei_extra_v2", # 結果儲存目錄
    "plot_grid_mapping_path": "grid_mapping_visualization_taipei.png", # 網格映射視覺化圖片路徑
    "train_split_ratio": 0.7, # 訓練集比例
    "val_split_ratio": 0.15,  # 驗證集比例
}

os.makedirs(CONFIG["save_dir"], exist_ok=True)
logger.info(f"結果將儲存於: {CONFIG['save_dir']}")

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if CONFIG["device"] == "cuda":
    torch.cuda.manual_seed_all(CONFIG["seed"])
logger.info(f"使用裝置: {CONFIG['device']}")

2025-05-12 13:25:00,509 - INFO - 結果將儲存於: results_ddpm_conditioned_flow_taipei_extra_v2
2025-05-12 13:25:00,510 - INFO - 使用裝置: cuda


In [ ]:
# --------------------------------------
# 數據處理相關
# --------------------------------------
def parse_lat_lon(column_name: str) -> tuple[float, float]:
    match = re.search(r'\(([\d.-]+),\s*([\d.-]+)\)', column_name)
    if match:
        return float(match.group(1)), float(match.group(2))
    raise ValueError(f"欄位名稱格式無效：{column_name}")

class PeopleFlowDatasetCondition(Dataset):
    def __init__(self,
                 df: pd.DataFrame, # 傳入 DataFrame 物件
                 config: Dict[str, Any],
                 mode: str = 'train',
                 # 驗證/測試模式下，由訓練資料集實例傳入
                 average_flow_map_dict: Optional[Dict[Tuple[int, int], np.ndarray]] = None,
                 norm_stats_flow: Optional[Dict[str, float]] = None,
                 sorted_flow_columns_from_train: Optional[List[str]] = None,
                 grid_idx_to_rc_map_from_train: Optional[Dict[int, Tuple[int,int]]] = None,
                 processed_extra_columns_from_train: Optional[List[str]] = None,
                 selected_sensor_info_from_train: Optional[List[Dict[str, Any]]] = None # <--- 新增此參數
                ):
        super().__init__()
        self.df_original = df.reset_index(drop=True)
        self.config = config
        self.mode = mode
        self.H = config["H"]
        self.W = config["W"]
        self.D = config.get("D", 1)
        self.image_channels = config.get("image_channels", 1)
        self.num_grid_cells = self.H * self.W

        # --- 時間解析 ---
        actual_datetime_col = '時間'
        if actual_datetime_col not in self.df_original.columns:
            raise ValueError(f"資料中未找到指定的日期時間欄位 '{actual_datetime_col}'。")
        df_datetime_processed = self.df_original.copy()
        df_datetime_processed[actual_datetime_col] = pd.to_datetime(df_datetime_processed[actual_datetime_col])
        self.hours_original_np = df_datetime_processed[actual_datetime_col].dt.hour.values
        self.day_of_week_original_np = df_datetime_processed[actual_datetime_col].dt.dayofweek.values

        # --- 處理額外資料 ---
        df_for_extra_processing = self.df_original.copy()
        if actual_datetime_col in df_for_extra_processing.columns:
            temp_dt_series = pd.to_datetime(df_for_extra_processing[actual_datetime_col])
            df_for_extra_processing['年'] = temp_dt_series.dt.year
            df_for_extra_processing['月'] = temp_dt_series.dt.month
            df_for_extra_processing['日'] = temp_dt_series.dt.day
            df_for_extra_processing['時'] = temp_dt_series.dt.hour
            df_for_extra_processing['weekday'] = temp_dt_series.dt.dayofweek
        else:
            logger.warning(f"df_for_extra_processing 中未找到時間欄位 '{actual_datetime_col}'，部分額外時間特徵可能無法生成。")

        self.extra_cols_list_definition = config.get("extra_features_definition_list", [
            "測站氣壓", "海平面氣壓", "氣溫", "露點溫度", "相對溼度", "風速", "最大陣風",
            "降水量", "降水時數", "日照時數", "全天空日射量", "能見度", "紫外線指數", "總雲量",
            "holiday", "weekday", "年", "月", "日", "時"
        ])
        wind_cols_to_process = []
        if '風向' in df_for_extra_processing.columns: wind_cols_to_process.append('風向')
        if '最大陣風風向' in df_for_extra_processing.columns: wind_cols_to_process.append('最大陣風風向')
        for col in wind_cols_to_process:
            df_for_extra_processing[f'sin_{col}'] = np.sin(np.deg2rad(df_for_extra_processing[col].astype(float)))
            df_for_extra_processing[f'cos_{col}'] = np.cos(np.deg2rad(df_for_extra_processing[col].astype(float)))
            if f'sin_{col}' not in self.extra_cols_list_definition: self.extra_cols_list_definition.append(f'sin_{col}')
            if f'cos_{col}' not in self.extra_cols_list_definition: self.extra_cols_list_definition.append(f'cos_{col}')
        self.extra_cols_list_definition = [col for col in self.extra_cols_list_definition if col not in ['風向', '最大陣風風向']]
        present_extra_cols = [col for col in self.extra_cols_list_definition if col in df_for_extra_processing.columns]
        df_extra_subset = df_for_extra_processing[present_extra_cols].copy()
        if "hoilday" in df_extra_subset.columns: # 修正可能的拼寫錯誤
            df_extra_subset.rename(columns={"hoilday": "holiday"}, inplace=True)
        cat_features = ['holiday']
        actual_cat_features = [col for col in cat_features if col in df_extra_subset.columns]
        if actual_cat_features:
            df_extra_subset[actual_cat_features] = df_extra_subset[actual_cat_features].astype(str)
            df_cat = pd.get_dummies(df_extra_subset[actual_cat_features], prefix=actual_cat_features, dummy_na=False)
        else:
            df_cat = pd.DataFrame(index=df_extra_subset.index)
        df_cont = df_extra_subset.drop(columns=actual_cat_features, errors='ignore')
        df_extra_processed = pd.concat([df_cont, df_cat], axis=1)

        if self.mode == 'train':
            self.processed_extra_columns = list(df_extra_processed.columns)
            self.processed_extra_data_np = df_extra_processed.fillna(0).values.astype(np.float32)
            logger.info(f"訓練集: 已處理 {len(self.processed_extra_columns)} 個額外特徵。")
        else:
            if processed_extra_columns_from_train is None:
                raise ValueError("驗證/測試模式，必須提供 processed_extra_columns_from_train。")
            self.processed_extra_columns = processed_extra_columns_from_train
            df_extra_processed = df_extra_processed.reindex(columns=self.processed_extra_columns, fill_value=0)
            self.processed_extra_data_np = df_extra_processed.fillna(0).values.astype(np.float32)
            logger.info(f"{self.mode} 資料集: 已處理額外特徵。")

        # --- 流量資料網格映射與平均值計算 ---
        if self.mode == 'train':
            all_flow_columns_with_coords = [c for c in self.df_original.columns if '(' in c and ')' in c]
            logger.info(f"從欄位名稱中找到 {len(all_flow_columns_with_coords)} 個可能的流量/座標欄位。")
            num_required_points = self.H * self.W
            if len(all_flow_columns_with_coords) < num_required_points:
                raise ValueError(
                    f"網格大小 ({self.H}x{self.W}={num_required_points}) 大於了可用的地理座標點數量 ({len(all_flow_columns_with_coords)}). "
                    "請減少 H*W 或提供更多座標點。"
                )
            all_column_info_list = []
            for col_name in all_flow_columns_with_coords:
                lon, lat = parse_lat_lon(col_name)
                all_column_info_list.append({'name': col_name, 'lon': lon, 'lat': lat})
            if len(all_column_info_list) < num_required_points:
                 raise ValueError(
                    f"成功解析的地理座標點數量 ({len(all_column_info_list)}) 不足所需的網格點 ({num_required_points})。"
                )
            all_coords_np = np.array([(info['lon'], info['lat']) for info in all_column_info_list])
            self.selected_sensor_info = []
            selected_real_coords_np_for_grid_def = None
            if len(all_column_info_list) > num_required_points:
                logger.info(f"座標點數量 ({len(all_column_info_list)}) 多於網格數 ({num_required_points}). "
                             f"將選擇最靠近地理中心的 {num_required_points} 個座標點進行映射。")
                geometric_center_lon = np.mean(all_coords_np[:, 0])
                geometric_center_lat = np.mean(all_coords_np[:, 1])
                distances_to_geometric_center = np.sqrt(
                    (all_coords_np[:, 0] - geometric_center_lon)**2 +
                    (all_coords_np[:, 1] - geometric_center_lat)**2
                )
                selected_indices = np.argsort(distances_to_geometric_center)[:num_required_points]
                self.selected_sensor_info = [all_column_info_list[i] for i in selected_indices]
                selected_real_coords_np_for_grid_def = all_coords_np[selected_indices]
            else:
                self.selected_sensor_info = all_column_info_list
                selected_real_coords_np_for_grid_def = all_coords_np
            logger.info(f"已選定 {len(self.selected_sensor_info)} 個感測器進行網格映射。")
            self.grid_target_coords, self.grid_idx_to_rc_map = self._define_target_grid_cells_hierarchical_style(selected_real_coords_np_for_grid_def)
            self.sorted_flow_columns = self._map_sensors_to_target_grid_hungarian(
                self.selected_sensor_info,
                selected_real_coords_np_for_grid_def,
                self.grid_target_coords
            )
            plot_path = os.path.join(self.config["save_dir"], self.config.get("plot_grid_mapping_path", "grid_mapping_visualization.png"))
            self._plot_grid_mapping( # 函數會從 self 獲取 selected_sensor_info
                self.grid_idx_to_rc_map,
                self.sorted_flow_columns,
                plot_path
            )
            self.average_flow_map_dict = self._calculate_average_flows()
            all_avg_flows_list = [flow for flow in self.average_flow_map_dict.values() if flow is not None]
            if not all_avg_flows_list:
                raise ValueError("訓練集中未計算出任何平均流量。無法計算流量標準化統計量。")
            all_avg_flows_np = np.stack(all_avg_flows_list)
            self.flow_mean_val = np.mean(all_avg_flows_np)
            self.flow_std_val = np.std(all_avg_flows_np)
            if self.flow_std_val < 1e-5: self.flow_std_val = 1e-5
            self.norm_stats_flow = {'mean': self.flow_mean_val, 'std': self.flow_std_val}
            logger.info(f"訓練集流量標準化統計量: 平均值={self.flow_mean_val:.4f}, 標準差={self.flow_std_val:.4f}")
        else:
            if not all([average_flow_map_dict, 
                        norm_stats_flow, 
                        sorted_flow_columns_from_train, 
                        grid_idx_to_rc_map_from_train, 
                        processed_extra_columns_from_train, # 這個之前就有
                        selected_sensor_info_from_train]): # <--- 加入對 selected_sensor_info_from_train 的檢查
                raise ValueError("驗證/測試模式下，必須提供 average_flow_map_dict, norm_stats_flow, "
                                 "sorted_flow_columns_from_train, grid_idx_to_rc_map_from_train, "
                                 "processed_extra_columns_from_train, selected_sensor_info_from_train。")
            
            self.average_flow_map_dict = average_flow_map_dict
            self.norm_stats_flow = norm_stats_flow
            self.flow_mean_val = self.norm_stats_flow['mean']
            self.flow_std_val = self.norm_stats_flow['std']
            self.sorted_flow_columns = sorted_flow_columns_from_train
            self.grid_idx_to_rc_map = grid_idx_to_rc_map_from_train
            self.processed_extra_columns = processed_extra_columns_from_train
            self.selected_sensor_info = selected_sensor_info_from_train # <--- 確保賦值
            
            logger.info(f"{self.mode} 資料集使用預計算的流量標準化統計量、欄位順序、網格映射和感測器資訊。")

    def _define_target_grid_cells_hierarchical_style(self, selected_real_coords_np: np.ndarray) -> Tuple[np.ndarray, Dict[int, Tuple[int, int]]]:
        if selected_real_coords_np.shape[0] != self.num_grid_cells:
            raise ValueError(f"selected_real_coords_np 應含 {self.num_grid_cells} 點, 得到 {selected_real_coords_np.shape[0]}。")
        logger.info("使用 'hierarchical' 風格定義目標網格中心點...")
        center_lon, center_lat = np.mean(selected_real_coords_np, axis=0)
        unique_lons = np.unique(selected_real_coords_np[:,0])
        unique_lats = np.unique(selected_real_coords_np[:,1])
        lon_diffs = np.diff(np.sort(unique_lons))
        lat_diffs = np.diff(np.sort(unique_lats))
        lon_step = np.median(lon_diffs[lon_diffs > 1e-6]) if len(lon_diffs[lon_diffs > 1e-6]) > 0 else 0.001
        lat_step = np.median(lat_diffs[lat_diffs > 1e-6]) if len(lat_diffs[lat_diffs > 1e-6]) > 0 else 0.001
        if lon_step <= 1e-6: lon_step = 0.001 # 避免步長為0
        if lat_step <= 1e-6: lat_step = 0.001 # 避免步長為0
        logger.info(f"目標網格中心: (lon:{center_lon:.4f}, lat:{center_lat:.4f}), 步長: (lon:{lon_step:.6f}, lat:{lat_step:.6f})")

        grid_targets = np.zeros((self.num_grid_cells, 2))
        idx_to_rc = {}
        idx = 0
        for r_idx in range(self.H):
            for c_idx in range(self.W):
                tlon = center_lon + (c_idx - (self.W-1)/2.0) * lon_step
                tlat = center_lat - (r_idx - (self.H-1)/2.0) * lat_step # 緯度通常由北往南增加索引
                grid_targets[idx, 0], grid_targets[idx, 1] = tlon, tlat
                idx_to_rc[idx] = (r_idx,c_idx)
                idx += 1
        return grid_targets, idx_to_rc

    def _map_sensors_to_target_grid_hungarian(self,
                                            sel_sensor_info_list: List[Dict[str,Any]],
                                            sel_coords_np: np.ndarray,
                                            grid_target_coords_np: np.ndarray
                                            ) -> List[str]:
        logger.info("使用匈牙利演算法將選定感測器映射到目標網格...")
        n_sensors = sel_coords_np.shape[0]
        n_targets = grid_target_coords_np.shape[0]
        if n_sensors != self.num_grid_cells or n_targets != self.num_grid_cells:
            raise ValueError(f"感測器數量 ({n_sensors}) 或目標網格點數量 ({n_targets}) "
                             f"必須等於 H*W ({self.num_grid_cells})。")
        if len(sel_sensor_info_list) != n_sensors:
             raise ValueError(f"sel_sensor_info_list 的長度 ({len(sel_sensor_info_list)}) 與 sel_coords_np ({n_sensors}) 不符。")
        costs = np.sqrt(np.sum((sel_coords_np[:, np.newaxis, :] - grid_target_coords_np[np.newaxis, :, :])**2, axis=2))
        assigned_real_indices, assigned_target_indices = linear_sum_assignment(costs)
        target_to_real_map = {t_idx: r_idx for r_idx, t_idx in zip(assigned_real_indices, assigned_target_indices)}
        sorted_flow_column_names = [""] * self.num_grid_cells
        for flat_target_idx in range(self.num_grid_cells):
            if flat_target_idx in target_to_real_map:
                real_idx_in_sel_list = target_to_real_map[flat_target_idx]
                sorted_flow_column_names[flat_target_idx] = sel_sensor_info_list[real_idx_in_sel_list]['name']
            else:
                raise Exception(f"目標網格索引 {flat_target_idx} 未分配到任何感測器。")
        logger.info(f"成功為網格排序 {len(sorted_flow_column_names)} 個流量欄位。")
        return sorted_flow_column_names

    def _calculate_average_flows(self) -> Dict[Tuple[int, int], np.ndarray]:
        """計算每個 (小時, 星期幾) 組合的平均流量圖。"""
        logger.info("計算 (小時, 星期幾) 平均流量圖...")
        avg_flows = {}
        for col in self.sorted_flow_columns:
            if col not in self.df_original.columns:
                raise ValueError(f"流量欄位 '{col}' 在 DataFrame 中未找到。")
        flow_data_grid_alltimes = self.df_original[self.sorted_flow_columns].values.astype(np.float32)
        grouping_df = pd.DataFrame({'hour': self.hours_original_np, 'day_of_week': self.day_of_week_original_np})
        for (hr, dow), group_indices in grouping_df.groupby(['hour', 'day_of_week']).groups.items():
            group_flows_for_condition = flow_data_grid_alltimes[group_indices]
            mean_flow_flat_for_condition = np.nanmean(group_flows_for_condition, axis=0)
            mean_flow_flat_for_condition[np.isnan(mean_flow_flat_for_condition)] = 0
            avg_flows[(hr, dow)] = mean_flow_flat_for_condition.reshape(self.H, self.W)
        if not avg_flows:
            logger.warning("未計算任何 (小時, 星期幾) 的平均流量。")
        logger.info(f"計算完成 {len(avg_flows)} 個 (小時, 星期幾) 條件的平均流量圖。")
        return avg_flows

    def _plot_grid_mapping(self,
                           grid_idx_to_rc_map: Dict[int, Tuple[int,int]],
                           sorted_flow_cols: List[str], 
                           save_path: str):
        """繪製並儲存簡化後的網格映射視覺化圖，僅顯示實際感測器點及其網格編號。"""
        plt.figure(figsize=(12, 12)) 
        plt.style.use('seaborn-v0_8-whitegrid')
        if not hasattr(self, 'selected_sensor_info') or not self.selected_sensor_info:
            logger.error("_plot_grid_mapping: self.selected_sensor_info 未定義或為空。")
            plt.close(); return
        if not hasattr(self, 'sorted_flow_columns') or not self.sorted_flow_columns:
            logger.error("_plot_grid_mapping: self.sorted_flow_columns 未定義或為空。")
            plt.close(); return
        if len(self.sorted_flow_columns) != self.num_grid_cells:
            logger.error(f"_plot_grid_mapping: sorted_flow_columns 長度 ({len(self.sorted_flow_columns)}) 與 num_grid_cells ({self.num_grid_cells}) 不符。")
            plt.close(); return

        selected_sensor_info_dict = {info['name']: (info['lon'], info['lat']) for info in self.selected_sensor_info}
        actual_sensor_lons = []
        actual_sensor_lats = []
        grid_labels_for_actual_sensors = []
        for flat_grid_idx in range(self.num_grid_cells):
            col_name = sorted_flow_cols[flat_grid_idx] 
            if col_name in selected_sensor_info_dict:
                lon, lat = selected_sensor_info_dict[col_name]
                actual_sensor_lons.append(lon)
                actual_sensor_lats.append(lat)
                r_map, c_map = grid_idx_to_rc_map.get(flat_grid_idx, (-1,-1))
                if r_map != -1: 
                    grid_labels_for_actual_sensors.append(f'[{r_map},{c_map}]')
                else:
                    grid_labels_for_actual_sensors.append(f'[{flat_grid_idx}]') 
                    logger.warning(f"未在 grid_idx_to_rc_map 中找到扁平網格索引 {flat_grid_idx} 的 (r,c) 映射。")
            else:
                logger.warning(f"欄位 {col_name} (來自 sorted_flow_cols，索引 {flat_grid_idx}) "
                               f"在 selected_sensor_info_dict 中未找到。跳過此點的繪製。")
        if not actual_sensor_lons:
            logger.warning("在 _plot_grid_mapping 中未能收集到任何實際感測器座標點進行繪製。")
            plt.close(); return
        plt.scatter(actual_sensor_lons, actual_sensor_lats,
                    c='blue', marker='o', s=50, alpha=0.7, label='網格點 (實際感測器位置)')
        for i in range(len(actual_sensor_lons)):
            if i < len(grid_labels_for_actual_sensors):
                 plt.text(actual_sensor_lons[i], actual_sensor_lats[i],
                         grid_labels_for_actual_sensors[i], 
                         fontsize=7, color='navy', ha='right', va='bottom') 
        plt.xlabel("經度 (Longitude)")
        plt.ylabel("緯度 (Latitude)")
        plt.title(f"網格映射 ({self.H}x{self.W})") 
        plt.grid(True, linestyle=':', alpha=0.6)
        plt.gca().set_aspect('equal', adjustable='box') 
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"簡化後的網格映射視覺化圖已儲存至 {save_path}")

    def __len__(self) -> int:
        return len(self.df_original)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int, int, torch.Tensor]:
        current_hour_original = self.hours_original_np[idx]
        current_dow_original = self.day_of_week_original_np[idx]
        target_avg_flow_np = self.average_flow_map_dict.get((current_hour_original, current_dow_original))
        if target_avg_flow_np is None:
            logger.warning(f"在 average_flow_map_dict 中找不到 ({current_hour_original}, {current_dow_original}) 的平均流量，將使用零值網格。")
            target_avg_flow_np = np.zeros((self.H, self.W), dtype=np.float32)
        if not hasattr(self, 'flow_mean_val') or not hasattr(self, 'flow_std_val'):
             raise AttributeError("flow_mean_val 或 flow_std_val 未在 Dataset 初始化時設定。")
        standardized_avg_flow_np = (target_avg_flow_np - self.flow_mean_val) / self.flow_std_val
        target_flow_tensor = torch.from_numpy(standardized_avg_flow_np).float()
        if self.D == 1 and self.image_channels == 1:
            target_flow_tensor = target_flow_tensor.unsqueeze(0).unsqueeze(0)
        elif self.image_channels > 1 or self.D > 1:
             target_flow_tensor = target_flow_tensor.unsqueeze(0) 
             if self.D > 1: 
                 target_flow_tensor = target_flow_tensor.repeat(self.D, 1, 1) 
             target_flow_tensor = target_flow_tensor.unsqueeze(0) 
             if target_flow_tensor.shape[1] != self.D or target_flow_tensor.shape[0] != self.image_channels:
                 logger.warning(f"target_flow_tensor shape {target_flow_tensor.shape} 與期望 C={self.image_channels}, D={self.D} 不符，請檢查 unsqueeze/repeat 邏輯")
        if not hasattr(self, 'processed_extra_data_np'):
            raise AttributeError("processed_extra_data_np 未在 Dataset 初始化時設定。")
        extra_data_row_tensor = torch.from_numpy(self.processed_extra_data_np[idx]).float()
        return target_flow_tensor, int(current_hour_original), int(current_dow_original), extra_data_row_tensor
    

In [ ]:
# ==============================================================================
# UNet3D, DDPM3D
# ==============================================================================

# UNet3D 建構模組及 UNet3D 類別的預留位置
class SinusoidalTimeEmbedding(nn.Module):
    """正弦時間嵌入"""
    def __init__(self, dim: int): super().__init__(); self.dim = dim
    def forward(self, t: torch.Tensor) -> torch.Tensor:
        device = t.device; half_dim = self.dim // 2; emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = t[:, None] * emb[None, :]; emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb

class DoubleConv3D(nn.Module):
    """(卷積3D -> BN -> SiLU) * 2"""
    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None, kernel_size: int = 3, padding: int = 1):
        super().__init__(); mid_channels = mid_channels or out_channels
        self.double_conv = nn.Sequential(
            nn.Conv3d(in_channels, mid_channels, kernel_size=kernel_size, padding=padding, bias=False), nn.BatchNorm3d(mid_channels), nn.SiLU(inplace=True),
            nn.Conv3d(mid_channels, out_channels, kernel_size=kernel_size, padding=padding, bias=False), nn.BatchNorm3d(out_channels), nn.SiLU(inplace=True))
    def forward(self, x: torch.Tensor) -> torch.Tensor: return self.double_conv(x)

class Down3D(nn.Module):
    """下採樣模組 (MaxPool3D -> DoubleConv3D)"""
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool3d(kernel_size=(1,2,2), stride=(1,2,2)), # 深度維度不壓縮
            DoubleConv3D(in_channels, out_channels)
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor: return self.maxpool_conv(x)

class Up3D(nn.Module):
    """上採樣模組"""
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__(); self.bilinear = bilinear
        if bilinear:
            self.up = nn.Upsample(scale_factor=(1,2,2), mode='trilinear', align_corners=True) # 深度維度不放大
            self.conv = DoubleConv3D(in_channels, out_channels, mid_channels=in_channels // 2)
        else:
            self.up = nn.ConvTranspose3d(in_channels, in_channels // 2, kernel_size=(1,2,2), stride=(1,2,2)) # 深度維度不放大
            self.conv = DoubleConv3D(in_channels, out_channels)
    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor: # x1 是上採樣的張量, x2 是殘差連接的張量
        x1 = self.up(x1)
        # 輸入大小: C D H W
        diffY = x2.size()[3] - x1.size()[3] # H
        diffX = x2.size()[4] - x1.size()[4] # W
        # 深度維度 (dim 2) 不需要填充
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, # W
                        diffY // 2, diffY - diffY // 2, # H
                        0, 0])                          # D
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class OutConv3D(nn.Module):
    """輸出卷積層 (1x1x1 Conv3D)"""
    def __init__(self, in_channels: int, out_channels: int): super().__init__(); self.conv = nn.Conv3d(in_channels, out_channels, kernel_size=1)
    def forward(self, x: torch.Tensor) -> torch.Tensor: return self.conv(x)

class UNet3D(nn.Module):
    """3D U-Net 模型，帶有正確的時間嵌入投影"""
    def __init__(self, input_image_channels: int, base_channels: int = 64, time_emb_dim: int = 256,
                 condition_encode_dim: Optional[int] = None, bilinear_upsample: bool = True, dropout_rate: float = 0.05):
        super().__init__()
        self.input_image_channels = input_image_channels
        self.condition_encode_dim = condition_encode_dim or 0

        # 共享的時間嵌入 MLP (輸出維度是 time_emb_dim)
        self.shared_time_mlp = nn.Sequential(
            SinusoidalTimeEmbedding(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim)
        )

        actual_in_channels = self.input_image_channels + self.condition_encode_dim
        
        # --- U-Net 結構 ---
        self.inc = DoubleConv3D(actual_in_channels, base_channels)
        self.down1 = Down3D(base_channels, base_channels * 2)
        self.down2 = Down3D(base_channels * 2, base_channels * 4)
        self.down3 = Down3D(base_channels * 4, base_channels * 8)
        factor = 2 if bilinear_upsample else 1
        self.down4 = Down3D(base_channels * 8, base_channels * 16 // factor) # Bottleneck 層的前一層
        self.dropout = nn.Dropout3d(dropout_rate) if dropout_rate > 0 else nn.Identity()

        self.up1 = Up3D(base_channels * 16, base_channels * 8 // factor, bilinear_upsample)
        self.up2 = Up3D(base_channels * 8, base_channels * 4 // factor, bilinear_upsample)
        self.up3 = Up3D(base_channels * 4, base_channels * 2 // factor, bilinear_upsample)
        self.up4 = Up3D(base_channels * 2, base_channels, bilinear_upsample)
        self.outc = OutConv3D(base_channels, self.input_image_channels)

        # --- 為每個需要添加時間嵌入的層級定義線性投影層 ---
        # 這些層的輸出維度與對應特徵圖的通道數匹配
        self.time_proj_inc = nn.Linear(time_emb_dim, base_channels)
        self.time_proj_down1 = nn.Linear(time_emb_dim, base_channels * 2)
        self.time_proj_down2 = nn.Linear(time_emb_dim, base_channels * 4)
        self.time_proj_down3 = nn.Linear(time_emb_dim, base_channels * 8)
        self.time_proj_bottleneck = nn.Linear(time_emb_dim, base_channels * 16 // factor) # 對應 down4 的輸出 (bottleneck)

        self.time_proj_up1 = nn.Linear(time_emb_dim, base_channels * 8 // factor)
        self.time_proj_up2 = nn.Linear(time_emb_dim, base_channels * 4 // factor)
        self.time_proj_up3 = nn.Linear(time_emb_dim, base_channels * 2 // factor)
        self.time_proj_up4 = nn.Linear(time_emb_dim, base_channels)

    def _add_time_embedding(self, x: torch.Tensor, t_emb_projected: torch.Tensor) -> torch.Tensor:
        # t_emb_projected 應該已經是 (N, C_feature_map) 的形狀
        t_emb_expanded = t_emb_projected.unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
        return x + t_emb_expanded

    def forward(self, x_t: torch.Tensor, time_steps: torch.Tensor, processed_condition: Optional[torch.Tensor] = None) -> torch.Tensor:
        # 首先計算共享的時間嵌入 (N, time_emb_dim)
        shared_t_emb = self.shared_time_mlp(time_steps)

        if processed_condition is not None:
            if x_t.shape[2:] != processed_condition.shape[2:]: # 檢查 D, H, W 是否一致
                raise ValueError(f"x_t DHW {x_t.shape[2:]} != processed_condition DHW {processed_condition.shape[2:]}")
            x_input = torch.cat((x_t, processed_condition), dim=1) # 沿通道維度合併
        else:
            x_input = x_t

        x1 = self.inc(x_input)
        x1 = self._add_time_embedding(x1, self.time_proj_inc(shared_t_emb))

        x2 = self.down1(x1)
        x2 = self._add_time_embedding(x2, self.time_proj_down1(shared_t_emb))

        x3 = self.down2(x2)
        x3 = self._add_time_embedding(x3, self.time_proj_down2(shared_t_emb))

        x4 = self.down3(x3)
        x4 = self._add_time_embedding(x4, self.time_proj_down3(shared_t_emb))

        x5 = self.down4(x4) # Bottleneck 特徵
        x5 = self._add_time_embedding(x5, self.time_proj_bottleneck(shared_t_emb)) # 使用對應的投影
        x5 = self.dropout(x5)

        # Decoder path
        x = self.up1(x5, x4) # x4 是來自 encoder 的 skip connection
        x = self._add_time_embedding(x, self.time_proj_up1(shared_t_emb))

        x = self.up2(x, x3) # x3 是來自 encoder 的 skip connection
        x = self._add_time_embedding(x, self.time_proj_up2(shared_t_emb))

        x = self.up3(x, x2) # x2 是來自 encoder 的 skip connection
        x = self._add_time_embedding(x, self.time_proj_up3(shared_t_emb))

        x = self.up4(x, x1) # x1 是來自 encoder 的 skip connection
        x = self._add_time_embedding(x, self.time_proj_up4(shared_t_emb))
        
        return self.outc(x)
def linear_beta_schedule(timesteps: int, beta_start: float, beta_end: float) -> torch.Tensor:
    """線性 beta 排程"""
    return torch.linspace(beta_start, beta_end, timesteps)

class DDPM3D(nn.Module):
    """3D Denoising Diffusion Probabilistic Model"""
    def __init__(self,
                 unet_model: UNet3D,
                 timesteps: int,
                 image_size: Tuple[int, int, int], # (D, H, W)
                 image_channels: int,
                 condition_input_channels: int, # 條件處理器輸入的原始通道數 (例如: 小時網格+星期網格 = 2)
                 condition_encode_dim: int,     # 條件處理器輸出的編碼維度
                 beta_start: float = 1e-4,
                 beta_end: float = 0.02,
                 device: str = "cpu"):
        super().__init__()
        self.model = unet_model # U-Net 模型
        self.timesteps = timesteps
        self.image_size_D, self.image_size_H, self.image_size_W = image_size # 儲存 D, H, W
        self.image_channels = image_channels
        self.device = device

        # --- 擴散排程參數 ---
        self.betas = linear_beta_schedule(timesteps, beta_start, beta_end).to(device)
        self.alphas = 1. - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, axis=0) # α_bar_t
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0) # α_bar_{t-1}
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - self.alphas_cumprod)
        self.posterior_variance = self.betas * (1. - self.alphas_cumprod_prev) / (1. - self.alphas_cumprod) # p(x_{t-1}|x_t, x_0) 的變異數

        # --- 條件處理器 (例如：將小時、星期幾網格編碼) ---
        # 輸入: (N, condition_input_channels, D, H, W)
        # 輸出: (N, condition_encode_dim, D, H, W)
        self.condition_processor = nn.Sequential(
            nn.Conv3d(condition_input_channels, condition_encode_dim // 2,
                      kernel_size=(1, 3, 3), padding=(0, 1, 1), bias=False), # 深度維度 kernel=1, padding=0
            nn.BatchNorm3d(condition_encode_dim // 2), nn.SiLU(),
            nn.Conv3d(condition_encode_dim // 2, condition_encode_dim,
                      kernel_size=(1, 3, 3), padding=(0, 1, 1), bias=False), # 深度維度 kernel=1, padding=0
            nn.BatchNorm3d(condition_encode_dim), nn.SiLU()
        ).to(device)

    def _extract(self, a: torch.Tensor, t: torch.Tensor, x_shape: Tuple[int, ...]) -> torch.Tensor:
        """從 a 中提取對應 t 時刻的值，並調整形狀以匹配 x_shape"""
        batch_size = t.shape[0]
        out = a.gather(-1, t) # (batch_size,)
        return out.reshape(batch_size, *((1,) * (len(x_shape) - 1))) # (batch_size, 1, 1, 1, 1)

    def q_sample(self, x_start: torch.Tensor, t: torch.Tensor, noise: Optional[torch.Tensor] = None) -> torch.Tensor:
        """前向擴散過程 (加噪)：q(x_t | x_0)"""
        if noise is None: noise = torch.randn_like(x_start)
        sact = self._extract(self.sqrt_alphas_cumprod, t, x_start.shape)
        soma_ct = self._extract(self.sqrt_one_minus_alphas_cumprod, t, x_start.shape)
        return sact * x_start + soma_ct * noise # x_t

    def _prepare_conditional_input_grids(self,
                                        hour_scalars_batch: torch.Tensor, # (N,) 原始 0-23
                                        day_scalars_batch: torch.Tensor,  # (N,) 原始 0-6
                                        ) -> torch.Tensor: # 輸出 (N, 2, D, H, W)
        """將純量的小時和星期幾轉換為正規化的網格輸入"""
        batch_size = hour_scalars_batch.shape[0]
        # 在此正規化純量值
        norm_hours = hour_scalars_batch.float().to(self.device) / 23.0 # 正規化到 [0, 1]
        norm_days = day_scalars_batch.float().to(self.device) / 6.0   # 正規化到 [0, 1]

        # 建立 HxW 的網格，每個網格的值相同
        hour_grids_list = [torch.full((self.image_size_H, self.image_size_W), norm_hours[i].item(), device=self.device, dtype=torch.float32) for i in range(batch_size)]
        day_grids_list = [torch.full((self.image_size_H, self.image_size_W), norm_days[i].item(), device=self.device, dtype=torch.float32) for i in range(batch_size)]

        hour_grids_t = torch.stack(hour_grids_list, dim=0).unsqueeze(1).unsqueeze(2) # (N,1,1,H,W)
        day_grids_t = torch.stack(day_grids_list, dim=0).unsqueeze(1).unsqueeze(2)   # (N,1,1,H,W)

        # 確保深度維度匹配 self.image_size_D (在此專案中 D=1)
        if self.image_size_D != 1: # 理論上此專案不會進入此分支
             hour_grids_t = hour_grids_t.repeat(1,1,self.image_size_D,1,1)
             day_grids_t = day_grids_t.repeat(1,1,self.image_size_D,1,1)

        return torch.cat((hour_grids_t, day_grids_t), dim=1) # (N, 2, D, H, W)

    def p_losses(self, x_start: torch.Tensor, t: torch.Tensor,
                 hour_scalars_batch: torch.Tensor, day_scalars_batch: torch.Tensor,
                 # extra_data_batch: torch.Tensor, # 目前未直接由條件處理器使用
                 noise: Optional[torch.Tensor] = None) -> torch.Tensor:
        """計算損失 (預測雜訊與真實雜訊的 MSE)"""
        if noise is None: noise = torch.randn_like(x_start)
        x_t = self.q_sample(x_start=x_start, t=t, noise=noise) # 得到加噪影像 x_t

        # 準備並處理條件輸入
        stacked_cond_grids = self._prepare_conditional_input_grids(hour_scalars_batch, day_scalars_batch) # (N, 2, D, H, W)
        processed_condition = self.condition_processor(stacked_cond_grids) # (N, C_cond_enc, D, H, W)

        predicted_noise = self.model(x_t, t, processed_condition) # U-Net 預測雜訊
        return F.mse_loss(noise, predicted_noise)

    @torch.no_grad()
    def p_sample(self, x_t: torch.Tensor, t_scalar: int, t_tensor_batch: torch.Tensor,
                 processed_conditions_batch: torch.Tensor) -> torch.Tensor:
        """逆向過程單步取樣：p(x_{t-1} | x_t)"""
        # t_tensor_batch 是 (batch_size,)，每個元素都是 t_scalar
        betas_t = self._extract(self.betas, t_tensor_batch, x_t.shape)
        sqrt_one_minus_alphas_cumprod_t = self._extract(self.sqrt_one_minus_alphas_cumprod, t_tensor_batch, x_t.shape)
        sqrt_recip_alphas_t = self._extract(torch.sqrt(1.0 / self.alphas), t_tensor_batch, x_t.shape) # 1/sqrt(α_t)

        # 使用 U-Net 預測雜訊
        predicted_noise = self.model(x_t, t_tensor_batch, processed_conditions_batch)
        # 計算 x_0_hat 的均值部分 (DDPM 公式)
        model_mean = sqrt_recip_alphas_t * (x_t - betas_t * predicted_noise / sqrt_one_minus_alphas_cumprod_t)

        if t_scalar == 0: # 最後一步，直接返回均值
            return model_mean
        else:
            posterior_variance_t = self._extract(self.posterior_variance, t_tensor_batch, x_t.shape)
            noise = torch.randn_like(x_t) # 加入隨機雜訊
            return model_mean + torch.sqrt(posterior_variance_t) * noise

    @torch.no_grad()
    def p_sample_loop(self, shape: Tuple[int,...], hour_scalars_batch: torch.Tensor, day_scalars_batch: torch.Tensor) -> torch.Tensor:
        """逆向過程完整取樣迴圈"""
        batch_size = shape[0]
        img = torch.randn(shape, device=self.device) # 從純雜訊 x_T 開始

        # 預先處理條件，因為在迴圈中條件是固定的
        stacked_cond_grids = self._prepare_conditional_input_grids(hour_scalars_batch, day_scalars_batch)
        processed_conditions = self.condition_processor(stacked_cond_grids) # (batch_size, C_cond_enc, D, H, W)

        for i in tqdm(reversed(range(0, self.timesteps)), desc="DDPM 取樣迴圈", total=self.timesteps, leave=False):
            t_tensor_batch = torch.full((batch_size,), i, device=self.device, dtype=torch.long)
            img = self.p_sample(img, i, t_tensor_batch, processed_conditions)
        return img # 返回生成的影像 x_0

    @torch.no_grad()
    def sample(self, batch_size: int, hour_scalars_batch: torch.Tensor, day_scalars_batch: torch.Tensor) -> torch.Tensor:
        """生成一批樣本"""
        # hour_scalars_batch, day_scalars_batch 應為 (batch_size,)
        s = (batch_size, self.image_channels, self.image_size_D, self.image_size_H, self.image_size_W)
        return self.p_sample_loop(s, hour_scalars_batch, day_scalars_batch)

In [84]:
# FID 函數 (get_activations, calculate_frechet_distance, calculate_fid)
def get_activations(images: torch.Tensor, model: nn.Module, device: str, batch_size_fid: int = 32) -> np.ndarray:
    """使用 Inception 模型提取影像特徵。"""
    model.eval()
    activations = []

    # 處理影像維度以符合 InceptionV3 輸入
    # images: (N, C, D, H, W)
    if images.shape[2] == 1: # D=1
        images_2d = images.squeeze(2) # (N, C, H, W)
    else: # D > 1, 取中間切片
        images_2d = images[:, :, images.shape[2]//2, :, :]
        logger.warning("影像深度 > 1，為 FID 取中間切片。")

    if images_2d.shape[1] == 1: # C=1, 複製為 3 通道
        images_2d = images_2d.repeat(1, 3, 1, 1)
    elif images_2d.shape[1] != 3 : # C != 1 且 C != 3, 取前 3 通道
        images_2d = images_2d[:,:3,:,:]
        logger.warning("影像通道數 != 1 或 3，為 FID 取前三通道。")

    # InceptionV3 需要 299x299 輸入
    transform_inception = transforms.Compose([
        transforms.Resize((299,299), antialias=True) # antialias=True 建議用於 PyTorch 1.7+
    ])

    num_batches = math.ceil(images_2d.shape[0] / batch_size_fid)
    for i in range(num_batches):
        batch = images_2d[i*batch_size_fid : (i+1)*batch_size_fid].to(device)
        batch = transform_inception(batch)
        with torch.no_grad():
            pred = model(batch) # InceptionV3 輸出
        if isinstance(pred, tuple): pred = pred[0] # 處理 InceptionV3 (非 aux_logits) 的輸出
        activations.append(pred.cpu().numpy())
    return np.concatenate(activations, axis=0)


def calculate_frechet_distance(mu1:np.ndarray, sigma1:np.ndarray, mu2:np.ndarray, sigma2:np.ndarray, eps:float=1e-6) -> float:
    """計算兩個多元高斯分佈之間的 Fréchet Distance。"""
    mu1,mu2 = np.atleast_1d(mu1), np.atleast_1d(mu2)
    sigma1,sigma2 = np.atleast_2d(sigma1), np.atleast_2d(sigma2)
    assert mu1.shape == mu2.shape, "均值向量的形狀必須匹配"
    assert sigma1.shape == sigma2.shape, "共變異數矩陣的形狀必須匹配"

    diff = mu1 - mu2
    # 計算 (sigma1 * sigma2) 的平方根
    covmean_sqrt, _ = scipy.linalg.sqrtm(sigma1.dot(sigma2), disp=False) # disp=False 避免印出警告
    if not np.isfinite(covmean_sqrt).all(): # 處理數值不穩定
        offset = np.eye(sigma1.shape[0]) * eps
        covmean_sqrt = scipy.linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))

    if np.iscomplexobj(covmean_sqrt): # 若結果是複數，取實部 (理論上應為實數)
        covmean_sqrt = covmean_sqrt.real

    return diff.dot(diff) + np.trace(sigma1) + np.trace(sigma2) - 2 * np.trace(covmean_sqrt)

def calculate_fid(real_acts:np.ndarray, gen_acts:np.ndarray)->float:
    """計算給定真實與生成影像特徵的 FID 分數。"""
    mu_real, sigma_real = real_acts.mean(axis=0), np.cov(real_acts, rowvar=False)
    mu_gen, sigma_gen = gen_acts.mean(axis=0), np.cov(gen_acts, rowvar=False)
    return calculate_frechet_distance(mu_real, sigma_real, mu_gen, sigma_gen)

In [85]:
def truncate_colormap(cmap, minval: float = 0.0, maxval: float = 1.0, n: int = 256):
    # (與 DDPM_3DUNet.ipynb 中的定義相同)
    new_cmap = mcolors.LinearSegmentedColormap.from_list(
        f'trunc({cmap.name},{minval:.2f},{maxval:.2f})',
        cmap(np.linspace(minval, maxval, n))
    )
    return new_cmap

def visualize_predictions_long_term(
                        generated_all_denorm_t: torch.Tensor, # (N, C, D, H, W) 反正規化後的生成數據
                        original_all_denorm_t: torch.Tensor,  # (N, C, D, H, W) 反正規化後的真實數據
                        config: Dict[str, Any],
                        sample_idx_to_plot: Optional[int] = 0, # 要繪製的特定樣本索引，None 表示繪製平均值
                        prefix: str = "test_eval" # 檔名前綴
                       ):
    """
    視覺化預測結果與真實值的比較 (針對 DDPM_Long-term.ipynb 的數據結構)。
    包含生成結果、真實數據、以及誤差（MSE、MAE、MAPE、SMAPE）的網格熱力圖。
    """
    save_dir = config["save_dir"]
    os.makedirs(save_dir, exist_ok=True)

    if generated_all_denorm_t.shape[2] > 1 or original_all_denorm_t.shape[2] > 1:
        logger.warning(f"visualize_predictions_long_term: 數據深度 > 1，將取 D 維度的平均值進行繪圖。")
        generated_all_denorm_t = torch.mean(generated_all_denorm_t, dim=2, keepdim=True)
        original_all_denorm_t = torch.mean(original_all_denorm_t, dim=2, keepdim=True)

    generated_squeezed = generated_all_denorm_t.squeeze(1).squeeze(1) # (N, H, W)
    original_squeezed = original_all_denorm_t.squeeze(1).squeeze(1)   # (N, H, W)

    H, W = generated_squeezed.shape[-2], generated_squeezed.shape[-1]

    if sample_idx_to_plot is None:
        gen_data_to_plot = torch.mean(generated_squeezed, dim=0).cpu().numpy() # (H, W)
        orig_data_to_plot = torch.mean(original_squeezed, dim=0).cpu().numpy() # (H, W)
        title_suffix = "all_samples_avg"
    elif sample_idx_to_plot < generated_squeezed.shape[0]:
        gen_data_to_plot = generated_squeezed[sample_idx_to_plot].cpu().numpy()
        orig_data_to_plot = original_squeezed[sample_idx_to_plot].cpu().numpy()
        title_suffix = f"sample_{sample_idx_to_plot}"
    else:
        logger.warning(f"sample_idx_to_plot {sample_idx_to_plot} 超出範圍，將繪製平均值。")
        gen_data_to_plot = torch.mean(generated_squeezed, dim=0).cpu().numpy()
        orig_data_to_plot = torch.mean(original_squeezed, dim=0).cpu().numpy()
        title_suffix = "all_samples_avg_fallback"

    epsilon = 1e-8 # 避免除以零
    mse_matrix = (gen_data_to_plot - orig_data_to_plot) ** 2
    mae_matrix = np.abs(gen_data_to_plot - orig_data_to_plot)
    mape_matrix = np.abs((orig_data_to_plot - gen_data_to_plot) / (np.abs(orig_data_to_plot) + epsilon)) * 100
    smape_matrix = np.abs(gen_data_to_plot - orig_data_to_plot) / ((np.abs(orig_data_to_plot) + np.abs(gen_data_to_plot))/2 + epsilon) * 100 



    overall_mse = np.mean(mse_matrix)
    overall_mae = np.mean(mae_matrix)
    overall_mape = np.mean(mape_matrix[np.isfinite(mape_matrix)])
    overall_smape = np.mean(smape_matrix[np.isfinite(smape_matrix)])


    # --- 修改開始：繪製6個子圖 ---
    fig, axes = plt.subplots(2, 3, figsize=(18, 10)) # 改成 2x3 的佈局

    # 圖 1: Generated
    im_gen = axes[0, 0].imshow(gen_data_to_plot, cmap='viridis')
    axes[0, 0].set_title(f'Generated ({title_suffix})')
    axes[0, 0].axis('off') # 隱藏座標軸
    fig.colorbar(im_gen, ax=axes[0, 0], fraction=0.046, pad=0.04)

    # 圖 2: Original
    im_orig = axes[0, 1].imshow(orig_data_to_plot, cmap='viridis')
    axes[0, 1].set_title(f'Original ({title_suffix})')
    axes[0, 1].axis('off')
    fig.colorbar(im_orig, ax=axes[0, 1], fraction=0.046, pad=0.04)

    # 圖 3: MSE
    im_mse = axes[0, 2].imshow(mse_matrix, cmap='hot')
    axes[0, 2].set_title(f'MSE Grid (Avg: {overall_mse:.0f})')
    axes[0, 2].axis('off')
    fig.colorbar(im_mse, ax=axes[0, 2], fraction=0.046, pad=0.04)

    # 圖 4: MAE
    im_mae = axes[1, 0].imshow(mae_matrix, cmap='hot')
    axes[1, 0].set_title(f'MAE Grid (Avg: {overall_mae:.0f})')
    axes[1, 0].axis('off')
    fig.colorbar(im_mae, ax=axes[1, 0], fraction=0.046, pad=0.04)

    # 圖 5: MAPE
    # MAPE 值可能差異很大，可以考慮使用 vmin 和 vmax 來設定顯示範圍
    vmax_mape = np.percentile(mape_matrix[np.isfinite(mape_matrix)], 98) if np.any(np.isfinite(mape_matrix)) else 100 # 取98百分位數作為上限，避免極端值影響
    im_mape = axes[1, 1].imshow(mape_matrix, cmap='cividis', vmin=0, vmax=vmax_mape if vmax_mape > 0 else 100)
    axes[1, 1].set_title(f'MAPE Grid (Avg: {overall_mape:.0f})')
    axes[1, 1].axis('off')
    fig.colorbar(im_mape, ax=axes[1, 1], fraction=0.046, pad=0.04)

    # 圖 6: SMAPE
    # SMAPE 值通常在 0-200% 或 0-100% (取決於定義)
    vmax_smape = np.percentile(smape_matrix[np.isfinite(smape_matrix)], 98) if np.any(np.isfinite(smape_matrix)) else 100
    im_smape = axes[1, 2].imshow(smape_matrix, cmap='cividis', vmin=0, vmax=vmax_smape if vmax_smape > 0 else 100) # SMAPE 範圍 0-100% 或 0-200%
    axes[1, 2].set_title(f'SMAPE Grid (Avg: {overall_smape:.0f})')
    axes[1, 2].axis('off')
    fig.colorbar(im_smape, ax=axes[1, 2], fraction=0.046, pad=0.04)
  

    plt.tight_layout()
    # 更改儲存的檔名以反映是六張圖的比較
    plt.savefig(os.path.join(save_dir, f'{prefix}_6maps_comparison_{title_suffix}.png'), dpi=300)
    plt.close(fig)

def plot_grid_with_error_long_term(
                        dataset_for_coords: Any, # 實際應為 PeopleFlowDatasetCondition 實例
                        error_metrics_grids: Dict[str, np.ndarray], # 例如: {'MSE': mse_grid, 'MAE': mae_grid, ...}
                        config: Dict[str, Any],
                        prefix: str = "test_eval"
                       ):
    """
    在地理座標上繪製每個網格點的平均誤差。
    標籤和標題已修改為英文。
    色彩映射修改為紅到黑，圖內數字為白色整數（MSE圖不顯示數字）。
    """
    logger = logging.getLogger(__name__) # 確保 logger 在函數作用域內可用
    save_dir = config["save_dir"]
    os.makedirs(save_dir, exist_ok=True)

    H, W = config["H"], config["W"]
    if not hasattr(dataset_for_coords, 'sorted_flow_columns') or \
       not hasattr(dataset_for_coords, 'grid_idx_to_rc_map') or \
       not hasattr(dataset_for_coords, 'selected_sensor_info'):
        logger.error("Dataset instance lacks necessary grid mapping information (sorted_flow_columns, grid_idx_to_rc_map, selected_sensor_info).")
        return

    selected_sensor_info_dict = {info['name']: (info['lon'], info['lat']) for info in dataset_for_coords.selected_sensor_info}

    actual_sensor_lons = []
    actual_sensor_lats = []
    valid_grid_indices_flat = []

    for flat_grid_idx in range(H * W):
        if flat_grid_idx < len(dataset_for_coords.sorted_flow_columns):
            col_name = dataset_for_coords.sorted_flow_columns[flat_grid_idx]
            if col_name in selected_sensor_info_dict:
                lon, lat = selected_sensor_info_dict[col_name]
                actual_sensor_lons.append(lon)
                actual_sensor_lats.append(lat)
                valid_grid_indices_flat.append(flat_grid_idx)
            else:
                logger.warning(f"plot_grid_with_error: Column {col_name} (expected at grid index {flat_grid_idx}) not found in selected_sensor_info_dict.")
        else:
            logger.warning(f"plot_grid_with_error: flat_grid_idx {flat_grid_idx} is out of bounds for sorted_flow_columns (length: {len(dataset_for_coords.sorted_flow_columns)}).")

    if not actual_sensor_lons:
        logger.error("plot_grid_with_error: Could not retrieve coordinates for any grid points.")
        return

    # 定義從紅色到黑色的色彩映射
    # 紅色 (低值) -> 黑色 (高值)
    cdict_red_to_black = {
        'red':   ((0.0, 1.0, 1.0),  # 在 0.0 (低值) 時，紅色為 1
                  (1.0, 0.0, 0.0)), # 在 1.0 (高值) 時，紅色為 0
        'green': ((0.0, 0.0, 0.0),  # 綠色始終為 0
                  (1.0, 0.0, 0.0)),
        'blue':  ((0.0, 0.0, 0.0),  # 藍色始終為 0
                  (1.0, 0.0, 0.0))
    }
    red_to_black_cmap = mcolors.LinearSegmentedColormap('RedToBlack', cdict_red_to_black)

    for metric_name, error_grid_flat in error_metrics_grids.items():
        if error_grid_flat.shape[0] != H*W :
            logger.error(f"Dimension of error_grid for metric {metric_name} ({error_grid_flat.shape}) is incorrect. Expected ({H*W},). Skipping plot.")
            continue

        error_values_for_plot = error_grid_flat[valid_grid_indices_flat]
        
        if len(error_values_for_plot) == 0:
            logger.warning(f"No valid error values to plot for metric {metric_name} after filtering by valid_grid_indices_flat. Skipping plot.")
            continue

        plt.figure(figsize=(12, 12))
        # 使用新的 red_to_black_cmap
        scatter = plt.scatter(actual_sensor_lons, actual_sensor_lats, c=error_values_for_plot, cmap=red_to_black_cmap, marker='s', s=100)
        plt.colorbar(scatter, label=metric_name)

        # 只有非 MSE 的指標圖才在網格上顯示數字
        if metric_name.upper() != 'MSE':
            for i in range(len(actual_sensor_lons)):
                val_to_text = error_values_for_plot[i]
                plt.text(actual_sensor_lons[i], actual_sensor_lats[i],
                         f'{val_to_text:.0f}', # 修改為顯示整數
                         fontsize=6, color='white', ha='center', va='center') # 修改文字顏色為白色

        plt.xlabel("Longitude")
        plt.ylabel("Latitude")
        plt.title(f"Geographic Grid Error Heatmap - {metric_name.upper()}")
        plt.grid(True, linestyle=':', alpha=0.6)
        plt.savefig(os.path.join(save_dir, f'{prefix}_grid_error_map_{metric_name.lower()}.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"Saved {metric_name} geographic grid error map.")
                        

In [86]:
# evaluate_model 函數
# (假設其定義與先前完整腳本相同，
# 但需傳遞純量小時/星期至 ddpm_model.sample)
@torch.no_grad()
def evaluate_model(ddpm_model: DDPM3D,
                   dataloader: DataLoader,
                   inception_model_fid: nn.Module,
                   config: Dict[str, Any],
                   max_samples_for_fid: Optional[int] = None # FID 計算的最大樣本數
                   ) -> Dict[str, float]:
    """評估 DDPM 模型，計算 MSE, MAE, MAPE, SMAPE, FID。"""
    ddpm_model.eval()
    inception_model_fid.eval()

    all_generated_samples_for_fid = [] # 儲存正規化的生成樣本 (FID用)
    all_original_samples_for_fid = []  # 儲存正規化的原始樣本 (FID用)

    # MSE, MAE, MAPE, SMAPE 在反正規化後的數值上操作
    all_generated_denorm_list = []
    all_original_denorm_list = []

    total_samples_processed_for_metrics = 0

    # 若未指定 FID 樣本數，則使用整個資料集
    max_fid_samples = max_samples_for_fid if max_samples_for_fid is not None else len(dataloader.dataset)

    pbar = tqdm(dataloader, desc="評估模型", leave=False)
    for batch_idx, (target_avg_flow_norm, hour_scalars, day_scalars, _) in enumerate(pbar): # _ 是 extra_data_rows
        current_batch_size = target_avg_flow_norm.shape[0]

        target_avg_flow_norm = target_avg_flow_norm.to(config["device"])

        # 生成流量圖 (正規化)
        generated_flow_norm = ddpm_model.sample(
            batch_size=current_batch_size,
            hour_scalars_batch=hour_scalars, # 直接傳遞，DDPM 內部處理裝置
            day_scalars_batch=day_scalars   # 直接傳遞
        ) # 輸出為 (N, 1, D, H, W)，已正規化

        # 反正規化以計算 MSE/MAE/MAPE/SMAPE
        if hasattr(dataloader.dataset, 'norm_stats_flow') and dataloader.dataset.norm_stats_flow is not None:
            mean_val = dataloader.dataset.norm_stats_flow['mean']
            std_val = dataloader.dataset.norm_stats_flow['std']
        else: # 若找不到標準化統計量 (理論上不應發生)
            logger.error("在資料集中找不到標準化統計量 (norm_stats_flow)。無法反正規化。")
            mean_val, std_val = 0, 1 # 預設為無操作

        generated_flow_denorm = generated_flow_norm * std_val + mean_val
        target_avg_flow_denorm = target_avg_flow_norm * std_val + mean_val

        all_generated_denorm_list.append(generated_flow_denorm.cpu())
        all_original_denorm_list.append(target_avg_flow_denorm.cpu())

        # 為 FID 收集正規化樣本
        if len(all_generated_samples_for_fid) * config.get("eval_batch_size", current_batch_size) < max_fid_samples : # 確保不超出 FID 樣本限制
             all_generated_samples_for_fid.append(generated_flow_norm.cpu())
             all_original_samples_for_fid.append(target_avg_flow_norm.cpu())

        total_samples_processed_for_metrics += current_batch_size
        # 此處不提早中斷，以便 MSE/MAE 等指標能在完整驗證/測試集上計算
        # FID 樣本數的限制主要影響 FID 計算部分

    # 串接所有反正規化的批次以計算指標
    if not all_generated_denorm_list: # 處理 dataloader 為空的情況
        logger.warning("評估期間未處理任何資料。返回零指標。")
        return {"mse": 0.0, "mae": 0.0, "mape": 0.0, "smape": 0.0, "fid": float('nan')} # FID 為 NaN

    generated_all_denorm_t = torch.cat(all_generated_denorm_list, dim=0)
    original_all_denorm_t = torch.cat(all_original_denorm_list, dim=0)

    # 在所有收集到的反正規化樣本上計算指標
    epsilon = 1e-8 # 用於 MAPE/SMAPE 避免除以零

    mse_total = F.mse_loss(generated_all_denorm_t, original_all_denorm_t).item()
    mae_total = F.l1_loss(generated_all_denorm_t, original_all_denorm_t).item()

    # MAPE 計算
    mape_total = torch.mean(torch.abs((original_all_denorm_t - generated_all_denorm_t) /
                                     (torch.abs(original_all_denorm_t) + epsilon))) * 100
    mape_total = mape_total.item()

    # SMAPE 計算 (常見定義: 200 * |pred - actual| / (|actual| + |pred| + epsilon))
    smape_numerator = torch.abs(generated_all_denorm_t - original_all_denorm_t)
    smape_denominator = torch.abs(original_all_denorm_t) + torch.abs(generated_all_denorm_t) + epsilon
    smape_total = torch.mean(200 * smape_numerator / smape_denominator)
    smape_total = smape_total.item()

    metrics = {"mse": mse_total, "mae": mae_total, "mape": mape_total, "smape": smape_total, "fid": float('nan')}

    # --- FID 計算 (使用正規化樣本) ---
    num_fid_samples_to_calc = 0
    if all_generated_samples_for_fid and all_original_samples_for_fid:
        generated_tensor_fid = torch.cat(all_generated_samples_for_fid, dim=0)[:max_fid_samples]
        original_tensor_fid = torch.cat(all_original_samples_for_fid, dim=0)[:max_fid_samples]
        num_fid_samples_to_calc = min(generated_tensor_fid.shape[0], original_tensor_fid.shape[0]) # 取實際收集到的較小者

        if num_fid_samples_to_calc > 1 : # 共變異數矩陣至少需要 2 個樣本
            logger.info(f"在 {num_fid_samples_to_calc} 個樣本上計算 FID...")
            act_generated = get_activations(generated_tensor_fid[:num_fid_samples_to_calc], inception_model_fid, config["device"], config["fid_batch_size"])
            act_original = get_activations(original_tensor_fid[:num_fid_samples_to_calc], inception_model_fid, config["device"], config["fid_batch_size"])

            if act_generated.shape[0] > 1 and act_original.shape[0] > 1: # 確保 get_activations 後仍有足夠樣本
                 metrics["fid"] = calculate_fid(act_original, act_generated)
                 logger.info(f"FID 計算完成: {metrics['fid']:.4f}")
            else:
                logger.warning("處理後，FID 計算的有效特徵不足。")
                metrics["fid"] = float('nan')
        else:
            logger.warning(f"收集或可用的 FID 計算樣本 ({num_fid_samples_to_calc}) 不足。")
            metrics["fid"] = float('nan')
    else:
        logger.warning("FID 的樣本列表為空。")
        metrics["fid"] = float('nan')

    logger.info("開始生成詳細的評估視覺化圖表...")

    # 1. 繪製模擬與實際熱力圖 (例如，第一個樣本或平均樣本)
    # generated_all_denorm_t 和 original_all_denorm_t 已經是反正規化的 Tensor
    # 確保它們在 CPU 上且為 NumPy 陣列 (如果函數內部需要)
    # visualize_predictions_long_term 函數內部會處理 .cpu().numpy()
    if generated_all_denorm_t is not None and original_all_denorm_t is not None:
        visualize_predictions_long_term(
            generated_all_denorm_t.clone().cpu(), # 傳遞副本以防意外修改
            original_all_denorm_t.clone().cpu(),
            config,
            sample_idx_to_plot=0, # 繪製測試集中的第一個樣本
            prefix=f"test_eval_sample0"
        )
        visualize_predictions_long_term(
            generated_all_denorm_t.clone().cpu(),
            original_all_denorm_t.clone().cpu(),
            config,
            sample_idx_to_plot=None, # 繪製所有測試樣本的平均值
            prefix=f"test_eval_avg"
        )

    # 初始化一個字典來儲存每個網格的指標，以防某些條件未滿足而未計算
    error_metrics_to_return_for_excel = {
        'MSE': np.array([]),
        'MAE': np.array([]),
        'MAPE': np.array([]),
        'SMAPE': np.array([])
    }

    # 2. 繪製每個網格點的平均誤差到地理座標圖上
    if hasattr(dataloader.dataset, 'H') and hasattr(dataloader.dataset, 'W'):
        H_dataset, W_dataset = dataloader.dataset.H, dataloader.dataset.W

        if generated_all_denorm_t.shape[1:3] == (config["image_channels"], config["D"]): # 確保維度正確
            mse_grid_flat = torch.mean((generated_all_denorm_t - original_all_denorm_t)**2, dim=[0,1,2]).cpu().numpy().flatten()
            mae_grid_flat = torch.mean(torch.abs(generated_all_denorm_t - original_all_denorm_t), dim=[0,1,2]).cpu().numpy().flatten()

            mape_grid_batch_flat = torch.abs((original_all_denorm_t - generated_all_denorm_t) / (torch.abs(original_all_denorm_t) + epsilon)) * 100
            mape_grid_flat = torch.mean(mape_grid_batch_flat, dim=[0,1,2]).cpu().numpy().flatten()

            smape_num_flat = torch.abs(generated_all_denorm_t - original_all_denorm_t)
            smape_den_flat = torch.abs(original_all_denorm_t) + torch.abs(generated_all_denorm_t) + epsilon
            smape_grid_batch_flat = 200 * smape_num_flat / smape_den_flat
            smape_grid_flat = torch.mean(smape_grid_batch_flat, dim=[0,1,2]).cpu().numpy().flatten()

            error_metrics_to_return_for_excel = { # 這將被回傳
                'MSE': mse_grid_flat,
                'MAE': mae_grid_flat,
                'MAPE': mape_grid_flat,
                'SMAPE': smape_grid_flat
            }
            # dataloader.dataset 應該是 PeopleFlowDatasetCondition 的實例
            plot_grid_with_error_long_term(dataloader.dataset, error_metrics_to_return_for_excel, config, prefix="test_eval")
        else:
            logger.warning("生成的張量維度與預期不符，跳過繪製地理網格誤差圖及每個網格的指標回傳。")
            # error_metrics_to_return_for_excel 將保持為包含空陣列的狀態
    else:
        logger.warning("Dataloader.dataset 未提供 H, W 屬性，跳過繪製地理網格誤差圖及每個網格的指標回傳。")
        # error_metrics_to_return_for_excel 將保持為包含空陣列的狀態

    logger.info("詳細評估視覺化圖表生成完畢。")
    # --- 視覺化結束 ---

    return metrics, error_metrics_to_return_for_excel

In [78]:
# 主訓練腳本
# (假設其定義與先前完整腳本相同，
# 但呼叫 ddpm.p_losses 的部分需傳遞純量小時/星期)
if __name__ == '__main__':
    logger.info("==========================================================")
    logger.info("    開始 DDPM 訓練 (額外資料未正規化，流量資料正規化)    ")
    logger.info("==========================================================")
    logger.info(f"組態設定: {json.dumps(CONFIG, indent=2)}")


    full_df = pd.read_csv(CONFIG["data_path"])
    logger.info(f"已載入資料: {CONFIG['data_path']}. 形狀: {full_df.shape}")



    # 資料分割
    total_len = len(full_df)
    train_len = int(CONFIG["train_split_ratio"] * total_len)
    val_len = int(CONFIG["val_split_ratio"] * total_len)
    test_len = total_len - train_len - val_len

    if train_len <= 0 or val_len <= 0 or test_len <= 0:
        raise ValueError(f"訓練/驗證/測試集分割錯誤。請檢查比例設定。"
                         f"Train: {train_len}, Val: {val_len}, Test: {test_len} out of {total_len}")

    df_shuffled = full_df.sample(frac=1, random_state=CONFIG["seed"]).reset_index(drop=True)
    train_df = df_shuffled[:train_len]
    val_df = df_shuffled[train_len : train_len + val_len]
    test_df = df_shuffled[train_len + val_len :]
    logger.info(f"資料分割: 訓練集={len(train_df)}, 驗證集={len(val_df)}, 測試集={len(test_df)}")


    logger.info("建立訓練資料集...")
    train_dataset = PeopleFlowDatasetCondition(train_df, CONFIG, mode='train')
    
    logger.info("建立驗證資料集...")
    val_dataset = PeopleFlowDatasetCondition(
        val_df,
        CONFIG,
        mode='val',
        average_flow_map_dict=train_dataset.average_flow_map_dict,
        norm_stats_flow=train_dataset.norm_stats_flow,
        sorted_flow_columns_from_train=train_dataset.sorted_flow_columns,
        grid_idx_to_rc_map_from_train=train_dataset.grid_idx_to_rc_map,
        processed_extra_columns_from_train=train_dataset.processed_extra_columns,
        selected_sensor_info_from_train=train_dataset.selected_sensor_info # <--- 確保傳遞此參數
    )

    logger.info("建立測試資料集...")
    test_dataset = PeopleFlowDatasetCondition(
        test_df,
        CONFIG,
        mode='test',
        average_flow_map_dict=train_dataset.average_flow_map_dict,
        norm_stats_flow=train_dataset.norm_stats_flow,
        sorted_flow_columns_from_train=train_dataset.sorted_flow_columns,
        grid_idx_to_rc_map_from_train=train_dataset.grid_idx_to_rc_map,
        processed_extra_columns_from_train=train_dataset.processed_extra_columns,
        selected_sensor_info_from_train=train_dataset.selected_sensor_info # <--- 確保傳遞此參數
    )


    train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG["eval_batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=CONFIG["eval_batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
    logger.info("DataLoaders 建立完成。")

    logger.info("初始化 UNet3D 模型...")
    unet = UNet3D(
        CONFIG["image_channels"],
        CONFIG["base_channels_unet"],
        CONFIG["time_emb_dim"],
        CONFIG["condition_encode_dim"],
        dropout_rate=CONFIG.get("unet_dropout_rate", 0.05) 
    ).to(CONFIG["device"]) 
    logger.info("初始化 DDPM3D 模型...")
    ddpm = DDPM3D(unet, CONFIG["timesteps"], (CONFIG["D"],CONFIG["H"],CONFIG["W"]), CONFIG["image_channels"],
                  CONFIG["condition_input_channels"], CONFIG["condition_encode_dim"],
                  CONFIG["beta_start"], CONFIG["beta_end"], CONFIG["device"]).to(CONFIG["device"])
    logger.info(f"UNet3D 參數數量: {sum(p.numel() for p in unet.parameters() if p.requires_grad):,}")
    logger.info(f"ConditionProcessor 參數數量: {sum(p.numel() for p in ddpm.condition_processor.parameters() if p.requires_grad):,}")

    # 優化器包含 U-Net 和條件處理器的參數
    optimizer = optim.AdamW(list(ddpm.model.parameters()) + list(ddpm.condition_processor.parameters()), lr=CONFIG["lr"])

    logger.info("載入 InceptionV3 以計算 FID...")
    # 保持 aux_logits=True 以匹配預訓練權重
    inception_fid = inception_v3(weights=Inception_V3_Weights.DEFAULT, aux_logits=True)

    inception_fid.fc = nn.Identity()

    # 按照您原始程式碼的風格，如果存在 AuxLogits，則將其設為 None
    if hasattr(inception_fid, 'AuxLogits') and inception_fid.AuxLogits is not None:
        inception_fid.AuxLogits = None

    inception_fid = inception_fid.to(CONFIG["device"])
    inception_fid.eval()
    logger.info("InceptionV3 載入完成。")


2025-05-11 22:44:40,404 - INFO - ==========================================================
2025-05-11 22:44:40,404 - INFO -     開始 DDPM 訓練 (額外資料未正規化，流量資料正規化)    
2025-05-11 22:44:40,405 - INFO - ==========================================================
2025-05-11 22:44:40,405 - INFO - 組態設定: {
  "data_path": "C:\\thesis\\code\\Taipei_CF\\all_merged.csv",
  "H": 20,
  "W": 20,
  "D": 1,
  "image_channels": 1,
  "condition_input_channels": 2,
  "condition_encode_dim": 16,
  "base_channels_unet": 64,
  "unet_dropout_rate": 0.1,
  "time_emb_dim": 256,
  "val_calculation_freq": 4,
  "timesteps": 1000,
  "beta_start": 0.0001,
  "beta_end": 0.02,
  "epochs": 128,
  "batch_size": 128,
  "lr": 0.001,
  "num_workers": 0,
  "device": "cuda",
  "seed": 42,
  "weight_decay": 1e-05,
  "lr_scheduler_factor": 0.5,
  "lr_scheduler_patience": 4,
  "lr_scheduler_min_lr": 1e-06,
  "early_stopping_patience": 8,
  "lr_reductions_before_val_decision": 3,
  "resume_from_checkpoint": true,
  "checkpoint_path"

In [87]:
optimizer = optim.AdamW(
    list(ddpm.model.parameters()) + list(ddpm.condition_processor.parameters()),
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"]
)

# --- 在開始訓練迴圈之前，定義 scheduler ---
# (修改) Scheduler 現在監控 avg_train_loss
scheduler = ReduceLROnPlateau(optimizer,
                              mode='min', # 訓練損失越小越好
                              factor=CONFIG["lr_scheduler_factor"],
                              patience=CONFIG["lr_scheduler_patience"],
                              min_lr=CONFIG["lr_scheduler_min_lr"])

start_epoch = 1 # 預設從 epoch 1 開始
best_loss_for_early_stopping_and_scheduler = float('inf')
best_val_loss_for_saving = float('inf')
best_val_loss_epoch = 0
metrics_hist = {'train_loss':[], 'val_loss':[], 'lr':[]}
early_stopping_counter = 0
last_calculated_avg_val_loss = float('inf')

checkpoint_filename = CONFIG.get("checkpoint_path", "best_ddpm_model_during_training.pth")
checkpoint_full_path = os.path.join(CONFIG["save_dir"], checkpoint_filename)

if CONFIG.get("resume_from_checkpoint", True) and os.path.exists(checkpoint_full_path):
    logger.info(f"找到檢查點: {checkpoint_full_path}，嘗試載入...")
    try:
        # 使用之前解決 UnpicklingError 的方法載入
        import numpy
        import pickle
        with torch.serialization.safe_globals([numpy, numpy.float32, numpy.float64, numpy.int32, numpy.int64]):
            chkpt = torch.load(checkpoint_full_path, map_location=CONFIG["device"], weights_only=False)
        
        # 載入模型狀態
        ddpm.load_state_dict(chkpt['ddpm_state_dict'])
        
        # 載入優化器和排程器狀態 (如果存在)
        if 'optimizer_state_dict' in chkpt:
            optimizer.load_state_dict(chkpt['optimizer_state_dict'])
            logger.info("已成功載入優化器狀態。")
        else:
            logger.warning("檢查點中未找到 'optimizer_state_dict'，優化器將從頭開始。")

        if 'scheduler_state_dict' in chkpt:
            scheduler.load_state_dict(chkpt['scheduler_state_dict'])
            logger.info("已成功載入排程器狀態。")
        else:
            logger.warning("檢查點中未找到 'scheduler_state_dict'，排程器將從頭開始。")

        # 恢復訓練進度相關的變數
        start_epoch = chkpt.get('epoch', 0) + 1 # 從下一個 epoch 開始
        
        # 恢復最佳驗證損失 (用於模型保存)
        best_val_loss_for_saving = chkpt.get('best_val_loss_for_saving', float('inf'))
        best_val_loss_epoch = chkpt.get('epoch', 0) # epoch ที่บันทึก best_val_loss

        # 恢復用於早停和 LR scheduler 的訓練損失 (如果您的邏輯是基於訓練損失)
        # 如果您的早停和 scheduler 基於驗證損失，則不需要下面這行
        # best_loss_for_early_stopping_and_scheduler = chkpt.get('best_train_loss_for_scheduler', float('inf')) 
        
        # 嘗試恢復 metrics_hist, early_stopping_counter, last_calculated_avg_val_loss
        # 這些通常在訓練迴圈內更新，如果檢查點是 epoch 結束時存的，可以考慮恢復
        # 但更簡單的做法是讓它們從頭開始記錄，或者只恢復 epoch 和損失，讓 scheduler 自己判斷
        # 為了簡單起見，這裡只恢復 epoch 和關鍵損失，其他讓訓練迴圈重新建立
        
        # 比較儲存的CONFIG和當前的CONFIG (可選，但建議)
        saved_config = chkpt.get('config', None)
        if saved_config:
            # 這裡可以加入更詳細的 CONFIG 比較邏輯
            if saved_config['H'] != CONFIG['H'] or saved_config['W'] != CONFIG['W']:
                logger.warning("警告：載入的檢查點 CONFIG 與當前 CONFIG 的網格尺寸不符！可能導致錯誤。")
            # ... 可以比較更多關鍵參數 ...
        
        logger.info(f"成功從 epoch {start_epoch-1} 的檢查點恢復訓練。將從 epoch {start_epoch} 開始。")
        logger.info(f"恢復的最佳驗證損失 (用於模型保存): {best_val_loss_for_saving:.5f} (在 epoch {best_val_loss_epoch})")

    except Exception as e:
        logger.error(f"載入檢查點 {checkpoint_full_path} 失敗: {e}。將從頭開始訓練。")
        start_epoch = 1 # 確保如果載入失敗，從頭開始
        # 重置其他可能被部分修改的變數
        best_loss_for_early_stopping_and_scheduler = float('inf')
        best_val_loss_for_saving = float('inf')
        best_val_loss_epoch = 0
        metrics_hist = {'train_loss':[], 'val_loss':[], 'lr':[]}
        early_stopping_counter = 0
        last_calculated_avg_val_loss = float('inf')
else:
    logger.info("未找到檢查點或未設定從檢查點恢復。將從頭開始訓練。")
    # start_epoch 等變數已是預設值

logger.info("開始訓練迴圈...")

# 用於早停和 scheduler 的最佳訓練損失
best_loss_for_early_stopping_and_scheduler = float('inf')

# 用於保存最佳模型的最佳驗證損失
best_val_loss_for_saving = float('inf')
best_val_loss_epoch = 0


metrics_hist = {'train_loss':[], 'val_loss':[], 'lr':[]}

early_stopping_patience = CONFIG["early_stopping_patience"]
early_stopping_counter = 0

# (新增) 用於儲存每個 epoch 的驗證損失，即使不是每個 epoch 都計算
# 如果某個 epoch 不計算，則沿用上一次計算的值或者標記為無效
# current_epoch_val_loss 將代表當前 epoch 計算出 (或沿用) 的驗證損失
# last_calculated_avg_val_loss 用於記錄最近一次 *實際計算* 的驗證損失
last_calculated_avg_val_loss = float('inf')


for epoch in range(1, CONFIG["epochs"] + 1):
    ddpm.train()
    total_train_loss = 0
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{CONFIG['epochs']} [訓練]", leave=False)
    for x_start, hour_s, day_s, _ in train_pbar:
        optimizer.zero_grad()
        x_start = x_start.to(CONFIG["device"])
        t = torch.randint(0, CONFIG["timesteps"], (x_start.shape[0],), device=CONFIG["device"]).long()
        loss = ddpm.p_losses(x_start, t, hour_s, day_s)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
        train_pbar.set_postfix({"損失": loss.item()})

    avg_train_loss = total_train_loss / len(train_loader)
    metrics_hist['train_loss'].append(avg_train_loss)

    # --- 計算驗證集損失 (圖像 MSE) ---
    # 初始化本 epoch 的驗證損失為 "未計算" 或上一次的值
    current_epoch_val_loss_calculated = False # 標記本 epoch 是否實際計算了 val loss
    avg_val_epoch_loss = last_calculated_avg_val_loss # 預設沿用，如果本 epoch 不計算

    if epoch % CONFIG.get("val_calculation_freq", 1) == 0: # (修改) 從 CONFIG 讀取頻率，預設為1 (每個epoch)
                                                          # 如果您仍想用 % 8 == 1, 請改為 (epoch -1) % 8 == 0 or epoch == 1
                                                          # 或者更簡單： CONFIG["val_calculation_freq"] = 8, if epoch % CONFIG["val_calculation_freq"] == 0 (或 1 如果從1開始)
        # 假設 CONFIG["val_calculation_freq"] = 8, 那就是每8個epoch計算一次
        # 若要與您原來的 if epoch % 8 == 1 一致 (即epoch 1, 9, 17...), 可以這樣：
        # if (epoch - 1) % CONFIG.get("val_calculation_freq", 8) == 0:
        # 為了簡化，這裡假設 val_calculation_freq 指的是間隔，例如每 val_calculation_freq 個 epoch 計算一次
        # 如果 CONFIG["val_calculation_freq"] = 1，則每個 epoch 都計算
        # 如果 CONFIG["val_calculation_freq"] = 8，則 epoch 8, 16, 24... 計算
        # 如果您希望是 epoch 1, 9, 17... ，則條件應為 (epoch - 1) % N == 0

        # 採用每 N 個 epoch 計算一次的邏輯，N 來自 CONFIG["val_calculation_freq"]
        # 預設 val_calculation_freq 為 1 (即每個 epoch 都計算驗證損失，以便最佳模型選擇更準確)
        # 如果您堅持之前的每8個epoch在第1,9,17...計算，請將此條件改回 if (epoch-1)%8 == 0:
        val_freq = CONFIG.get("val_calculation_freq", 1) # 預設每個epoch都計算
        if epoch == 1 or (epoch % val_freq == 0) : # 在第一個epoch和之後每val_freq個epoch計算
            current_epoch_val_loss_calculated = True
            ddpm.eval()
            total_val_epoch_loss_for_period = 0
            num_val_samples_processed = 0
            avg_val_epoch_loss = float('inf') # 重置為inf，如果驗證集為空則保持inf

            if len(val_loader.dataset) > 0:
                with torch.no_grad():
                    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch}/{CONFIG['epochs']} [驗證損失計算]", leave=False)
                    for val_x_start, val_hour_s, val_day_s, _ in val_pbar:
                        val_x_start = val_x_start.to(CONFIG["device"])
                        generated_flow_norm = ddpm.sample(
                            batch_size=val_x_start.shape[0],
                            hour_scalars_batch=val_hour_s,
                            day_scalars_batch=val_day_s
                        )
                        if not hasattr(train_dataset, 'norm_stats_flow') or train_dataset.norm_stats_flow is None:
                            raise ValueError("train_dataset.norm_stats_flow 未定義或為 None，無法進行反正規化。")
                        mean_val = train_dataset.norm_stats_flow['mean']
                        std_val = train_dataset.norm_stats_flow['std']
                        generated_flow_denorm = generated_flow_norm * std_val + mean_val
                        target_avg_flow_denorm = val_x_start * std_val + mean_val
                        batch_val_loss = F.mse_loss(generated_flow_denorm, target_avg_flow_denorm).item()
                        total_val_epoch_loss_for_period += batch_val_loss * val_x_start.shape[0]
                        num_val_samples_processed += val_x_start.shape[0]

                if num_val_samples_processed > 0:
                    avg_val_epoch_loss = total_val_epoch_loss_for_period / num_val_samples_processed
                    last_calculated_avg_val_loss = avg_val_epoch_loss # 更新最近 *實際計算* 的驗證損失
                else:
                    logger.warning(f"Epoch {epoch}: 驗證集為空，無法計算驗證損失。")
                    # avg_val_epoch_loss 保持 float('inf')
            else:
                 logger.warning(f"Epoch {epoch}: 驗證集為空，跳過驗證損失計算。")
                 # avg_val_epoch_loss 保持 float('inf')
    
    metrics_hist['val_loss'].append(avg_val_epoch_loss) # 記錄當前epoch的驗證損失（可能是新算的，也可能是沿用的）
    
    # (修改) 更新學習率，基於 avg_train_loss
    scheduler.step(avg_train_loss)
    current_lr = optimizer.param_groups[0]['lr']
    metrics_hist['lr'].append(current_lr)
    
    val_loss_display = f"{avg_val_epoch_loss:.5f}" if avg_val_epoch_loss != float('inf') else "N/A"
    if current_epoch_val_loss_calculated:
        val_loss_display += " (Calculated)"
    else:
        val_loss_display += " (Carried Over)"

    logger.info(f"Epoch {epoch}: Train Loss: {avg_train_loss:.5f} | Val Loss (Best Model Metric): {val_loss_display} | LR: {current_lr:.8f}")

    # --- 早停邏輯，基於 avg_train_loss ---
    if avg_train_loss < best_loss_for_early_stopping_and_scheduler:
        best_loss_for_early_stopping_and_scheduler = avg_train_loss
        early_stopping_counter = 0 # 重置早停計數器
    else:
        early_stopping_counter += 1
        logger.info(f"訓練損失未改善 (current: {avg_train_loss:.5f} vs best for ES: {best_loss_for_early_stopping_and_scheduler:.5f})，早停計數: {early_stopping_counter}/{early_stopping_patience}")
        if early_stopping_counter >= early_stopping_patience:
            logger.info(f"早停機制觸發於 Epoch {epoch} (基於訓練損失)。")
            break # 跳出訓練迴圈

    # --- 儲存最佳模型，基於 avg_val_epoch_loss ---
    # 只有當本 epoch 實際計算了驗證損失，並且該損失有效時，才考慮更新最佳模型
    if current_epoch_val_loss_calculated and avg_val_epoch_loss != float('inf'):
        if avg_val_epoch_loss < best_val_loss_for_saving:
            best_val_loss_for_saving = avg_val_epoch_loss
            best_val_loss_epoch = epoch
            save_path = os.path.join(CONFIG["save_dir"], "best_ddpm_model_during_training.pth") # 檔名保持不變或按需更改
            torch.save({
                'epoch': epoch,
                'ddpm_state_dict': ddpm.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_val_loss_for_saving': best_val_loss_for_saving, # 記錄的是驗證損失
                'train_loss_at_best_val': avg_train_loss, # 記錄此時的訓練損失
                'config': CONFIG,
                'norm_stats_flow': train_dataset.norm_stats_flow,
                'sorted_flow_columns': train_dataset.sorted_flow_columns,
                'grid_idx_to_rc_map': train_dataset.grid_idx_to_rc_map,
                'processed_extra_columns': train_dataset.processed_extra_columns,
            }, save_path)
            logger.info(f"已儲存新的最佳模型 (Epoch {best_val_loss_epoch} based on Val Loss: {best_val_loss_for_saving:.5f}) 至 {save_path}")

logger.info("訓練完成。")
if epoch < CONFIG["epochs"]:
    logger.info(f"訓練因早停而提前結束於 Epoch {epoch}。")

logger.info(f"訓練過程中，用於早停和LR調度的最低訓練損失是: {best_loss_for_early_stopping_and_scheduler:.5f}")
if best_val_loss_for_saving != float('inf'):
    logger.info(f"訓練過程中，用於模型選擇的最低驗證損失發生在 epoch {best_val_loss_epoch}，Val Loss (MSE): {best_val_loss_for_saving:.5f}")
else:
    logger.info("訓練過程中，未計算或未記錄到有效的最低驗證損失用於模型保存。")

2025-05-12 13:25:05,861 - INFO - 找到檢查點: results_ddpm_conditioned_flow_taipei_extra_v2\best_ddpm_model_during_training.pth，嘗試載入...


2025-05-12 13:25:06,452 - INFO - 已成功載入優化器狀態。
2025-05-12 13:25:06,453 - INFO - 已成功載入排程器狀態。
2025-05-12 13:25:06,453 - INFO - 成功從 epoch 8 的檢查點恢復訓練。將從 epoch 9 開始。
2025-05-12 13:25:06,453 - INFO - 恢復的最佳驗證損失 (用於模型保存): 13525.10308 (在 epoch 8)
2025-05-12 13:25:06,454 - INFO - 開始訓練迴圈...
2025-05-12 13:25:13,752 - INFO - Epoch 1: Train Loss: 0.00717 | Val Loss (Best Model Metric): N/A (Carried Over) | LR: 0.00001563
2025-05-12 13:25:21,093 - INFO - Epoch 2: Train Loss: 0.00790 | Val Loss (Best Model Metric): N/A (Carried Over) | LR: 0.00001563
2025-05-12 13:25:21,093 - INFO - 訓練損失未改善 (current: 0.00790 vs best for ES: 0.00717)，早停計數: 1/8
2025-05-12 13:25:28,357 - INFO - Epoch 3: Train Loss: 0.00756 | Val Loss (Best Model Metric): N/A (Carried Over) | LR: 0.00001563
2025-05-12 13:25:28,357 - INFO - 訓練損失未改善 (current: 0.00756 vs best for ES: 0.00717)，早停計數: 2/8


KeyboardInterrupt: 

In [89]:
# --- 所有 epoch 訓練完成後，載入最佳模型並進行最終評估 ---
logger.info("載入訓練過程中驗證損失最低的模型以進行最終測試集評估...")
best_model_path = os.path.join(CONFIG["save_dir"], "best_ddpm_model_during_training.pth")

if not os.path.exists(best_model_path):
    logger.error(f"找不到在訓練過程中儲存的最佳模型檔案: {best_model_path}。將使用最後一個 epoch 的模型進行評估。")
    # 如果沒有找到最佳模型（例如，如果上面的儲存邏輯有問題或被跳過），
    # final_ddpm 會是訓練結束時的 ddpm 狀態。
    # 也可以選擇在這裡 raise Error 或採取其他策略。
    final_ddpm_for_eval = ddpm # 使用最後一個 epoch 的模型
else:
    chkpt = torch.load(best_model_path, map_location=CONFIG["device"], weights_only=False)
    cfg_chkpt = chkpt.get('config', CONFIG)

    # 重新初始化模型結構以載入狀態字典
    final_unet_for_eval = UNet3D(
        cfg_chkpt["image_channels"],
        cfg_chkpt["base_channels_unet"],
        cfg_chkpt["time_emb_dim"],
        cfg_chkpt["condition_encode_dim"]
    ).to(CONFIG["device"])

    final_ddpm_for_eval = DDPM3D(
        final_unet_for_eval,
        cfg_chkpt["timesteps"],
        (cfg_chkpt["D"], cfg_chkpt["H"], cfg_chkpt["W"]),
        cfg_chkpt["image_channels"],
        cfg_chkpt["condition_input_channels"],
        cfg_chkpt["condition_encode_dim"],
        beta_start=cfg_chkpt.get("beta_start", CONFIG["beta_start"]),
        beta_end=cfg_chkpt.get("beta_end", CONFIG["beta_end"]),
        device=CONFIG["device"]
    )
    final_ddpm_for_eval.load_state_dict(chkpt['ddpm_state_dict'])
    logger.info(f"從 {best_model_path} 載入最佳模型 (Epoch {chkpt.get('epoch', '未知')}) 完成。")

# 使用 test_loader 和載入的最佳模型 (final_ddpm_for_eval) 進行最終評估
logger.info("在測試集上評估載入的最佳模型...")
# 確保 inception_fid 模型已定義和載入
if 'inception_fid' not in locals() or inception_fid is None:
    logger.info("重新載入 InceptionV3 以計算 FID (因為可能在訓練迴圈中未持續保持)...")
    inception_fid = inception_v3(weights=Inception_V3_Weights.DEFAULT, aux_logits=True)
    inception_fid.fc = nn.Identity()
    if hasattr(inception_fid, 'AuxLogits') and inception_fid.AuxLogits is not None:
        inception_fid.AuxLogits = None
    inception_fid = inception_fid.to(CONFIG["device"])
    inception_fid.eval()
    logger.info("InceptionV3 載入完成。")

test_metrics, per_grid_test_metrics = evaluate_model(
        final_ddpm_for_eval, 
        test_loader, 
        inception_fid, 
        CONFIG, 
        CONFIG["fid_num_samples"]
    )
logger.info(f"最終測試結果: MSE:{test_metrics['mse']:.5f}|MAE:{test_metrics['mae']:.5f}|MAPE:{test_metrics['mape']:.2f}%|SMAPE:{test_metrics['smape']:.2f}%|FID:{test_metrics['fid']:.3f}")

# 儲存最終測試指標
with open(os.path.join(CONFIG["save_dir"], "final_test_metrics.json"),'w') as f:
    json.dump(test_metrics, f, indent=4)
with open(os.path.join(CONFIG["save_dir"], "final_test_metrics.txt"),'w') as f:
    f.write(f"FINAL TEST METRICS:\nDate: {pd.Timestamp.now(tz='Asia/Taipei')}\n")
    for k,v in test_metrics.items():
        f.write(f"{k.upper()}: {v:.6f}\n")

logger.info("開始準備匯出 Excel 檔案的詳細指標...")
H_test = CONFIG["H"]
W_test = CONFIG["W"]
num_grid_cells_test = H_test * W_test

excel_data_rows = []

# test_dataset 可以從 test_loader 獲得
current_test_dataset = test_loader.dataset 

# 準備感測器資訊以便快速查找經緯度
sensor_info_lookup = {info['name']: {'lon': info['lon'], 'lat': info['lat']}
                        for info in current_test_dataset.selected_sensor_info}

for flat_idx in range(num_grid_cells_test):
    grid_rc = current_test_dataset.grid_idx_to_rc_map.get(flat_idx, (-1, -1)) # (row, col)
    
    lon, lat = np.nan, np.nan # 預設為 NaN
    if flat_idx < len(current_test_dataset.sorted_flow_columns):
        col_name = current_test_dataset.sorted_flow_columns[flat_idx]
        if col_name in sensor_info_lookup:
            lon = sensor_info_lookup[col_name]['lon']
            lat = sensor_info_lookup[col_name]['lat']
        else:
            logger.warning(f"Excel匯出：在 sensor_info_lookup 中找不到欄位 {col_name} (網格索引 {flat_idx}) 的經緯度。")
    else:
        logger.warning(f"Excel匯出：網格索引 {flat_idx} 超出 sorted_flow_columns 的範圍。")

    row_data = {
        '網格座標_R': grid_rc[0] if grid_rc[0] != -1 else '', # 網格橫座標
        '網格座標_C': grid_rc[1] if grid_rc[1] != -1 else '', # 網格縱座標
        '經度': lon,
        '緯度': lat,
        'MSE': per_grid_test_metrics.get('MSE')[flat_idx] if per_grid_test_metrics.get('MSE') is not None and flat_idx < len(per_grid_test_metrics.get('MSE')) else np.nan,
        'MAE': per_grid_test_metrics.get('MAE')[flat_idx] if per_grid_test_metrics.get('MAE') is not None and flat_idx < len(per_grid_test_metrics.get('MAE')) else np.nan,
        'MAPE': per_grid_test_metrics.get('MAPE')[flat_idx] if per_grid_test_metrics.get('MAPE') is not None and flat_idx < len(per_grid_test_metrics.get('MAPE')) else np.nan,
        'SMAPE': per_grid_test_metrics.get('SMAPE')[flat_idx] if per_grid_test_metrics.get('SMAPE') is not None and flat_idx < len(per_grid_test_metrics.get('SMAPE')) else np.nan,
        'FID': 'N/A' # FID 通常不是針對每個網格單元計算的
    }
    excel_data_rows.append(row_data)

# 準備平均指標列 (最後一列)
average_row_data = {
    '網格座標_R': '整體平均',
    '網格座標_C': '',
    '經度': '',
    '緯度': '',
    'MSE': test_metrics.get('mse', np.nan),
    'MAE': test_metrics.get('mae', np.nan),
    'MAPE': test_metrics.get('mape', np.nan),
    'SMAPE': test_metrics.get('smape', np.nan),
    'FID': test_metrics.get('fid', np.nan) # 全域 FID
}
excel_data_rows.append(average_row_data)

df_excel = pd.DataFrame(excel_data_rows)

# 定義 Excel 中的欄位順序
excel_column_order = ['網格座標_R', '網格座標_C', '經度', '緯度', 'MSE', 'MAE', 'MAPE', 'SMAPE', 'FID']
df_excel = df_excel[excel_column_order]

excel_filename = "final_test_metrics_detailed.xlsx"
excel_save_path = os.path.join(CONFIG["save_dir"], excel_filename)

try:
    df_excel.to_excel(excel_save_path, index=False, sheet_name='詳細測試指標')
    logger.info(f"詳細測試指標已成功匯出至 Excel 檔案: {excel_save_path}")
except Exception as e:
    logger.error(f"匯出 Excel 檔案失敗: {e}")

# 繪製訓練歷史圖表 (只包含訓練損失和驗證損失)
num_train_epochs_recorded = len(metrics_hist.get('train_loss', []))
num_val_epochs_recorded = len(metrics_hist.get('val_loss', []))
num_epochs_to_plot = min(num_train_epochs_recorded, num_val_epochs_recorded)

if num_epochs_to_plot > 0:
    ep_rng_plot = range(1, num_epochs_to_plot + 1)
    train_loss_plot = metrics_hist['train_loss'][:num_epochs_to_plot]
    val_loss_plot = metrics_hist['val_loss'][:num_epochs_to_plot]

    plt.figure(figsize=(10, 5))
    plt.style.use('seaborn-v0_8-darkgrid')
    plt.plot(ep_rng_plot, train_loss_plot, label='Training Loss')
    plt.plot(ep_rng_plot, val_loss_plot, label='Validation Loss (MSE)')
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f'Training and Validation Loss History (up to {num_epochs_to_plot} epochs)')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["save_dir"], "training_loss_history_plot.png"))
    plt.close()
    logger.info(f"已儲存訓練和驗證損失歷史圖表 (繪製了 {num_epochs_to_plot} 個 epochs)。")
else:
    logger.info("沒有足夠的數據來繪製訓練和驗證損失歷史圖表 (可能由於訓練提早中斷)。")

logger.info("================ 腳本執行完成 ================")

2025-05-12 13:37:16,088 - INFO - 載入訓練過程中驗證損失最低的模型以進行最終測試集評估...
2025-05-12 13:37:16,958 - INFO - 從 results_ddpm_conditioned_flow_taipei_extra_v2\best_ddpm_model_during_training.pth 載入最佳模型 (Epoch 8) 完成。
2025-05-12 13:37:16,958 - INFO - 在測試集上評估載入的最佳模型...
2025-05-12 13:46:01,432 - INFO - 在 128 個樣本上計算 FID...     
2025-05-12 13:46:04,283 - INFO - FID 計算完成: 0.7241
2025-05-12 13:46:04,284 - INFO - 開始生成詳細的評估視覺化圖表...
2025-05-12 13:46:06,518 - INFO - Saved MSE geographic grid error map.
2025-05-12 13:46:06,981 - INFO - Saved MAE geographic grid error map.
2025-05-12 13:46:07,444 - INFO - Saved MAPE geographic grid error map.
2025-05-12 13:46:07,863 - INFO - Saved SMAPE geographic grid error map.
2025-05-12 13:46:07,863 - INFO - 詳細評估視覺化圖表生成完畢。
2025-05-12 13:46:07,864 - INFO - 最終測試結果: MSE:13823.16016|MAE:60.15802|MAPE:6.04%|SMAPE:5.95%|FID:0.724
2025-05-12 13:46:07,865 - INFO - 開始準備匯出 Excel 檔案的詳細指標...
2025-05-12 13:46:08,028 - INFO - 詳細測試指標已成功匯出至 Excel 檔案: results_ddpm_conditioned_flow_taipei_extra